In [2]:
import pandas as pd

def calculate_and_display_lcoe():
    # Base Financial Parameters
    discount_rate = 0.08  # 8%
    hours_per_year = 8760

    # 1. Define Input Parameters for Display and Calculation
    # Note: BESS CapEx is converted to $/kW based on a 5-hour system ($250/kWh * 5 = $1250/kW)
    # Natural Gas and Uranium fuels are pre-calculated with their respective heat rates into $/MWh
    input_data = [
        {'Technology': 'ERCOT Grid (345 kV)', 'CapEx ($/kW)': 150, 'Fixed O&M ($/kW-yr)': 5, 'Var O&M ($/MWh)': 0, 'Fuel ($/MWh)': 75.00, 'Cap Factor': 1.00, 'Lifespan (Yrs)': 20},
        {'Technology': 'Utility Solar PPA', 'CapEx ($/kW)': 0, 'Fixed O&M ($/kW-yr)': 0, 'Var O&M ($/MWh)': 0, 'Fuel ($/MWh)': 35.00, 'Cap Factor': 0.28, 'Lifespan (Yrs)': 20},
        {'Technology': 'RICE (Natural Gas)', 'CapEx ($/kW)': 1200, 'Fixed O&M ($/kW-yr)': 0, 'Var O&M ($/MWh)': 15, 'Fuel ($/MWh)': 29.75, 'Cap Factor': 0.90, 'Lifespan (Yrs)': 20},
        {'Technology': 'Enhanced Geothermal', 'CapEx ($/kW)': 4500, 'Fixed O&M ($/kW-yr)': 115, 'Var O&M ($/MWh)': 0, 'Fuel ($/MWh)': 0.00, 'Cap Factor': 0.95, 'Lifespan (Yrs)': 30},
        {'Technology': 'Small Modular Reactor', 'CapEx ($/kW)': 7500, 'Fixed O&M ($/kW-yr)': 120, 'Var O&M ($/MWh)': 0, 'Fuel ($/MWh)': 8.32, 'Cap Factor': 0.95, 'Lifespan (Yrs)': 40},
        {'Technology': 'Microturbines', 'CapEx ($/kW)': 1500, 'Fixed O&M ($/kW-yr)': 0, 'Var O&M ($/MWh)': 20, 'Fuel ($/MWh)': 40.25, 'Cap Factor': 0.90, 'Lifespan (Yrs)': 20},
        {'Technology': 'Hybrid BESS (LCOS)', 'CapEx ($/kW)': 1250, 'Fixed O&M ($/kW-yr)': 10, 'Var O&M ($/MWh)': 0, 'Fuel ($/MWh)': 0.00, 'Cap Factor': 0.208, 'Lifespan (Yrs)': 15}
        # BESS Cap Factor: (5 hours * 365 days) / 8760 = 0.208
    ]

    df_inputs = pd.DataFrame(input_data)

    print("TABLE 1: LCOE Input Parameters (2026 Basis)")
    print("-" * 115)
    print(df_inputs.to_string(index=False))
    print("\n" + "=" * 115 + "\n")

    # 2. Calculate LCOE / LCOS
    results = []

    for row in input_data:
        tech = row['Technology']
        capacity_kw = 1000  # 1 MW reference system

        # Annual Energy Output (MWh)
        if tech == 'Hybrid BESS (LCOS)':
            annual_energy_mwh = 5 * 365  # 5 MWh discharged once per day
        else:
            annual_energy_mwh = (capacity_kw / 1000) * hours_per_year * row['Cap Factor']

        # Annual Costs
        capex_total = capacity_kw * row['CapEx ($/kW)']
        fixed_om_annual = capacity_kw * row['Fixed O&M ($/kW-yr)']
        var_om_annual = annual_energy_mwh * row['Var O&M ($/MWh)']
        fuel_annual = annual_energy_mwh * row['Fuel ($/MWh)']

        annual_op_cost = fixed_om_annual + var_om_annual + fuel_annual

        # Discounted Cash Flow Calculation (Present Value)
        pv_costs = capex_total
        pv_energy = 0

        for year in range(1, row['Lifespan (Yrs)'] + 1):
            discount_factor = (1 + discount_rate)**year
            pv_costs += annual_op_cost / discount_factor
            pv_energy += annual_energy_mwh / discount_factor

        lcoe = pv_costs / pv_energy
        results.append({'Technology': tech, 'Calculated LCOE ($/MWh)': round(lcoe, 2)})

    df_outputs = pd.DataFrame(results)

    print("TABLE 2: Calculated LCOE Results (8% Discount Rate)")
    print("-" * 55)
    print(df_outputs.to_string(index=False))

# Execute the function
if __name__ == "__main__":
    calculate_and_display_lcoe()

TABLE 1: LCOE Input Parameters (2026 Basis)
-------------------------------------------------------------------------------------------------------------------
           Technology  CapEx ($/kW)  Fixed O&M ($/kW-yr)  Var O&M ($/MWh)  Fuel ($/MWh)  Cap Factor  Lifespan (Yrs)
  ERCOT Grid (345 kV)           150                    5                0         75.00       1.000              20
    Utility Solar PPA             0                    0                0         35.00       0.280              20
   RICE (Natural Gas)          1200                    0               15         29.75       0.900              20
  Enhanced Geothermal          4500                  115                0          0.00       0.950              30
Small Modular Reactor          7500                  120                0          8.32       0.950              40
        Microturbines          1500                    0               20         40.25       0.900              20
   Hybrid BESS (LCOS)       